In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as mpl
import numpy as np
import time, os, json

mpl.rcParams['figure.figsize'] = (14, 5)
mpl.rcParams['figure.dpi'] = 110
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False


In [3]:
# Guys, use this notebook to make the graphs youll put in your article. Follow the article-template.html file under articles. 
# For the graphs to work, you're going to: 
#1. Turn your graphs into a json. You do this by running the export_for_web() function, specifying it the path of: 
# /articles/data/article-name/your_title.json
#2 Then go into your html, and edit the data-src class with your path. Also, if the chart is an aggregate of many 
# languages, use the aggregate tag in data-chart, if not, use the separate tag. If there is
#another kind of chart you need made, tell me and I'll add it for u



CACHE_DIR = 'data'
os.makedirs(CACHE_DIR, exist_ok=True)

LANGUAGES = {
    'English': 'en', 'French': 'fr', 'German': 'de',
    'Italian': 'it', 'Russian': 'ru', 'Spanish': 'es',
}
LANG_COLORS = {
    'English': '#1f77b4', 'French': '#d62728', 'German': '#2ca02c',
    'Italian': '#ff7f0e', 'Russian': '#9467bd', 'Spanish': '#8c564b',
}

def fetch_ngram(query, corpus='en', start_year=1800, end_year=2019, smoothing=3):
    safe = query.replace(' ', '_').replace('/', '-')[:60]
    cache_path = os.path.join(CACHE_DIR, f'{safe}_{corpus}_{start_year}_{end_year}.json')
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            raw = json.load(f)
    else:
        r = requests.get(
            'https://books.google.com/ngrams/json',
            params={'content': query, 'year_start': start_year,
                    'year_end': end_year, 'corpus': corpus, 'smoothing': smoothing},
            timeout=15
        )
        r.raise_for_status()
        raw = r.json()
        with open(cache_path, 'w') as f:
            json.dump(raw, f)
        time.sleep(0.5)
    if not raw:
        return pd.DataFrame({'year': range(start_year, end_year + 1)})
    years = list(range(start_year, end_year + 1))
    df = pd.DataFrame({'year': years})
    for item in raw:
        df[item['ngram']] = item['timeseries']
    return df

def fetch_across_languages(title_per_lang, start_year=1800, end_year=2019):
    results = {}
    for lang, term in title_per_lang.items():
        df = fetch_ngram(term, corpus=LANGUAGES[lang], start_year=start_year, end_year=end_year)
        col = [c for c in df.columns if c != 'year']
        s = df.set_index('year')[col[0]] if col else pd.Series(0.0, index=range(start_year, end_year + 1))
        s.index.name = 'year'
        results[lang] = s
    return results

def aggregate_across_languages(lang_series):
    parts = [s / s.max() for s in lang_series.values() if s.max() > 0]
    return sum(parts) if parts else None

def add_events(ax, events):
    if not events:
        return
    ymax = ax.get_ylim()[1]
    for year, label in events.items():
        ax.axvline(year, color='grey', linestyle=':', linewidth=1)
        ax.text(year + 0.5, ymax * 0.95, label, fontsize=7, color='grey', rotation=90, va='top')

def export_for_web(series_dict, events, save_path):
    years = list(next(iter(series_dict.values())).index.astype(int))
    payload = {
        'years': years,
        'series': {label: s.values.tolist() for label, s in series_dict.items()},
        'events': {str(k): v for k, v in (events or {}).items()},
    }
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, 'w') as f:
        json.dump(payload, f)

def plot_languages(lang_series, title, publication_year=None, events=None, save_path=None):
    fig, axes = mpl.subplots(1, 2, figsize=(16, 5))
    ax = axes[0]
    for lang, s in lang_series.items():
        ax.plot(s.index, s.values * 1e6, color=LANG_COLORS[lang], label=lang, linewidth=1.5)
    if publication_year:
        ax.axvline(publication_year, color='black', linestyle='--', linewidth=1,
                   label=f'Published ({publication_year})')
    ax.set_title(f'{title}\nFrequency per language (per million words)')
    ax.set_xlabel('Year'); ax.set_ylabel('Freq. per million words'); ax.legend(fontsize=8)
    add_events(ax, events)
    ax2 = axes[1]
    agg = aggregate_across_languages(lang_series)
    if agg is not None:
        ax2.fill_between(agg.index, agg.values, alpha=0.25, color='steelblue')
        ax2.plot(agg.index, agg.values, color='steelblue', linewidth=2)
    if publication_year:
        ax2.axvline(publication_year, color='black', linestyle='--', linewidth=1)
    ax2.set_title(f'{title}\nAggregated cross-language popularity index')
    ax2.set_xlabel('Year'); ax2.set_ylabel('Normalized aggregate')
    add_events(ax2, events)
    mpl.tight_layout()
    if save_path:
        mpl.savefig(save_path, dpi=150, bbox_inches='tight')
    mpl.show()

def plot_separate(lang_series, title, publication_year=None, events=None):
    fig, ax = mpl.subplots(figsize=(14, 5))
    for lang, s in lang_series.items():
        ax.plot(s.index, s.values * 1e6, color=LANG_COLORS[lang], label=lang, linewidth=1.5)
    if publication_year:
        ax.axvline(publication_year, color='black', linestyle='--', linewidth=1,
                   label=f'Published ({publication_year})')
    ax.set_title(f'{title}\nFrequency per language (per million words)')
    ax.set_xlabel('Year'); ax.set_ylabel('Freq. per million words'); ax.legend(fontsize=8)
    add_events(ax, events)
    mpl.tight_layout()
    mpl.show()

def plot_aggregate(lang_series, title, publication_year=None, events=None):
    fig, ax = mpl.subplots(figsize=(14, 5))
    agg = aggregate_across_languages(lang_series)
    if agg is not None:
        ax.fill_between(agg.index, agg.values, alpha=0.25, color='steelblue')
        ax.plot(agg.index, agg.values, color='steelblue', linewidth=2)
    if publication_year:
        ax.axvline(publication_year, color='black', linestyle='--', linewidth=1,
                   label=f'Published ({publication_year})')
    ax.set_title(f'{title}\nAggregated cross-language popularity index')
    ax.set_xlabel('Year'); ax.set_ylabel('Normalized aggregate')
    add_events(ax, events)
    mpl.tight_layout()
    mpl.show()

def plot_comparison(series_dict, title, events=None, save_path=None):
    fig, ax = mpl.subplots(figsize=(14, 5))
    colors = mpl.rcParams['axes.prop_cycle'].by_key()['color']
    for i, (label, s) in enumerate(series_dict.items()):
        ax.plot(s.index, s.values * 1e6, label=label, color=colors[i % len(colors)], linewidth=1.8)
    ax.set_title(title); ax.set_xlabel('Year')
    ax.set_ylabel('Freq. per million words'); ax.legend(fontsize=8)
    add_events(ax, events)
    mpl.tight_layout()
    if save_path:
        mpl.savefig(save_path, dpi=150, bbox_inches='tight')
        json_path = os.path.splitext(save_path)[0] + '.json'
        export_for_web(series_dict, events, json_path)
    mpl.show()

def fetch_series(query, corpus='en', start_year=1800, end_year=2019):
    """Fetch a single ngram as a year-indexed Series. Returns zeros if no data."""
    df = fetch_ngram(query, corpus=corpus, start_year=start_year, end_year=end_year)
    col = [c for c in df.columns if c != 'year']
    if col:
        return df.set_index('year')[col[0]]
    s = pd.Series(0.0, index=pd.Index(range(start_year, end_year + 1), name='year'))
    print(f'  WARNING: no data for "{query}" in corpus={corpus}')
    return s

print('Setup complete.')



Setup complete.


In [14]:
dq_titles = {
    'English': 'Don Quixote',
    'French':  'Don Quichotte',
    'German':  'Don Quijote',
    'Italian': 'Don Chisciotte',
    'Russian': 'Don Quixote',
    'Spanish': 'Don Quixote',
    'Spanish': 'Don Quijote',
}
dq_events = {
    1605: 'Part I published',
    1615: 'Part II published',
    1616: 'Cervantes dies\n(same day as Shakespeare)',
    1620: 'First English\ntranslation (Shelton)',
    1755: 'Smollett English\ntranslation',
    1808: 'Napoleonic invasion\nof Spain',
    1863: 'Dore illustrations\n(cultural revival)',
    1898: 'Spanish-American War\n/ Generation of 98',
    1914: 'WWI begins',
    1936: 'Spanish Civil War\nbegins',
    1939: 'Spanish Civil War ends\n/ Franco dictatorship',
    1965: 'Man of La Mancha\nmusical premieres',
    1975: 'Franco dies\n/ Spanish democracy',
    2005: '400th anniversary\nPart I',
    2015: '400th anniversary\nPart II',
}
dq_lang_data = fetch_across_languages(dq_titles, start_year=1600)
print('Fetched DQ data across', len(dq_lang_data), 'languages')


Fetched DQ data across 6 languages


In [15]:
six_and_five_comparison = {
    "The Great Gatsby":                     fetch_series('The Great Gatsby'),
    "Brave New World":                  fetch_series('Brave New World'),
    "Jane Eyre":                            fetch_series('Jane Eyre'),
    "Little Women":                     fetch_series('Little Women'),
    "Moby Dick":                        fetch_series('Moby Dick'),
    "Nineteen Eighty-Four":             fetch_series('Nineteen Eighty-Four'),
    "Pride and Prejudice":                  fetch_series('Pride and Prejudice'),
    "To Kill a Mockingbird":                fetch_series('To Kill a Mockingbird'),
    "Don Quixote":    fetch_series("Don Quixote")
}


publication_dates = {
    1813: 'Pride and Prejudice (Austen 1813)',
    1925: "The Great Gatsby",
    1932: "Brave New World",
    1847: "Jane Eyre",
    1868: "Little women",
    1851: "Moby dick",
    1949: "Nineteen eighty-four",
    1813: "Pride and prejudice",
    1951: "The catcher in the rye",
    1960: "To kill a mockingbird",
}

export_for_web(six_and_five_comparison, publication_dates, './articles/data/don-quixote/don-quijote-comparison.json')

Pride and Prejudice:

In [20]:
pp_titles = {
    'English': 'Pride and Prejudice',
    'French':  'Orgueil et Prejuge',
    'German':  'Stolz und Vorurteil',
    'Italian': 'Orgoglio e pregiudizio',
    'Russian': 'Pride and Prejudice',
    'Spanish': 'Orgullo y prejuicio',
}
pp_events = {
    1813: 'P&P published',
    1817: 'Austen dies',
    1833: 'First illus. edition',
    1870: 'Austen memoir pub.\n(sparked revival)',
    1894: 'Hugh Thomson illus.\nedition (major revival)',
    1914: 'WWI begins',
    1939: 'WWII begins',
    1948: 'First BBC TV adapt.',
    1967: 'BBC adaptation',
    1980: 'BBC adapt. (E. Garvie)',
    1995: 'BBC miniseries\n(Colin Firth)',
    2003: 'Bridget Jones film\n(P&P retelling)',
    2005: 'Joe Wright film',
    2013: 'P&P 200th anniversary',
}
pp_lang_data = fetch_across_languages(pp_titles)
print('Fetched P&P data across', len(pp_lang_data), 'languages')

export_for_web(pp_lang_data, pp_events, './articles/data/pride-and-prejudice/pride-and-prejudice.json')

Fetched P&P data across 6 languages


In [26]:
austen_works = {
    'Pride and Prejudice':   fetch_series('Pride and Prejudice'),
    'Sense and Sensibility': fetch_series('Sense and Sensibility'),
    'Northanger Abbey':      fetch_series('Northanger Abbey'),
    'Mansfield Park': fetch_series('Mansfield Park'),
}
austen_events = {
    1811: 'S&S published',
    1813: 'P&P published',
    1814: 'Mansfield Park published',
    1817: 'Austen dies\nNorthanger Abbey published',
    1948: 'First P&P BBC TV adapt.',
    1967: 'P&P BBC adaptation',
    1971: 'S&S BBC miniseries',
    1980: 'P&P BBC adapt. (E. Garvie)',
    1981: 'S&S BBC miniseries',
    1983: 'Mansfield Park BBC miniseries',
    1987: 'Northanger Abbey Giles Foster film',
    1995: 'P&P BBC miniseries (Colin Firth)\n S&S Emma Thompson film',
    1999: 'Mansfield Park Patricia Rozema film',
    2003: 'Bridget Jones film\n(P&P retelling)',
    2005: 'P&P Joe Wright film',
    2007: 'Northanger Abbey Jon Jones film\nMansfield Park ITV movie',
    2008: 'S&S BBC miniseries'
}
export_for_web(austen_works, austen_events, './articles/data/pride-and-prejudice/austen-works.json')

Little Women:

In [32]:
lw_titles = {
    'English': 'Little Women',
    'French':  'Les Quatre Filles du docteur March',
    'German':  'Little Women',
    'Italian': 'Piccole donne',
    'Russian': 'Маленькие женщины',
    'Spanish': 'Mujercitas'
}

lw_events = {
    1868: 'First volume published',
    1869: 'Second volume published',
    1888: 'Alcott dies',
    1917: 'First film adaptation\n(silent film)',
    1949: 'Hollywood film adaptation',
    1968: 'Novel centenary',
    1970: 'Second wave feminism\n(novel themes resonate)',
    1978: 'BBC miniseries adaptation',
    1994: 'Hollywood film adaptation\n(Winona Ryder)',
    2005: 'Broadway musical adaptation',
    2019: 'Greta Gerwig film adaptation\n(Saoirse Ronan)',
    2022: 'Broadway revival'
}
lw_lang_data = fetch_across_languages(lw_titles)
export_for_web(lw_lang_data, lw_events, './articles/data/little-women/little-women.json')

In [34]:
alcott_works = {
    'Little Women':   fetch_series('Little Women'),
    'Little Men':   fetch_series('Little Men'),
    'Eight Cousins':   fetch_series('Eight Cousins'),
    'An Old-Fashioned Girl':   fetch_series('An Old-Fashioned Girl'),
}

alcott_events = {
    1868: 'Little Women first volume published',
    1869: 'Little Women second volume published',
    1869: 'An Old-Fashioned Girl published',
    1871: 'Little Men published',
    1875: 'Eight Cousins published',
    1888: 'Alcott dies',
}

export_for_web(alcott_works, alcott_events, './articles/data/little-women/alcott-works.json')

In [33]:
lw_non_classics = {
    'Little Women':   fetch_series('Little Women'),
    'The Story of Avis': fetch_series('The Story of Avis'),
    'A Girl of the Limberlost': fetch_series('A Girl of the Limberlost'),
}

lw_non_classics_events = {
    1868: 'Little Women first volume published',
    1869: 'Little Women second volume published',
    1877: 'The Story of Avis published',
    1888: 'Alcott dies',
    1909: 'A Girl of the Limberlost published'
}

export_for_web(lw_non_classics, lw_non_classics_events, './articles/data/little-women/little-women-non-classics.json')

In [35]:
lw_classics = {
    'Little Women':   fetch_series('Little Women'),
    'Jane Eyre': fetch_series('Jane Eyre'),
    'Middlemarch': fetch_series('Middlemarch'),
}

lw_classics_events = {
    1847: 'Jane Eyre published',
    1868: 'Little Women first volume published',
    1869: 'Little Women second volume published',
    1871: 'Middlemarch published'
}

export_for_web(lw_classics, lw_classics_events, './articles/data/little-women/little-women-classics.json')

Brave New World:

In [37]:
bnw_titles = {
    'English': 'Brave New World',
    'French':  'Le Meilleur des mondes',
    'German':  'Schöne neue Welt',
    'Italian': 'Il mondo nuovo',
    'Russian': 'О дивный новый мир',
    'Spanish': 'Un mundo feliz',
}
bnw_events = {
    1932: 'Brave New World\npublished',
    1939: 'WWII begins',
    1945: 'WWII ends',
    1946: 'Huxley foreword\npublished',
    1949: '1984 published;\nrenewal of interest',
    1953: 'Stalin dies;\nCold War peaks',
    1958: 'BNW Revisited\npublished',
    1963: 'Sexual revolution /\npsychedelic era',
    1980: 'Reagan/Thatcher era',
    1998: 'TV film adaptation',
    2020: 'Peacock TV series',
}

bnw_lang_data = fetch_across_languages(bnw_titles, start_year=1900)
export_for_web(bnw_lang_data, bnw_events, './articles/data/brave-new-world/brave-new-world.json')

In [38]:
huxley_works = {
    'Crome Yellow (1921)':        fetch_series('Crome Yellow',        start_year=1900),
    'Point Counter Point (1928)': fetch_series('Point Counter Point', start_year=1900),
    'Brave New World (1932)':     fetch_series('Brave New World',     start_year=1900),
    'Eyeless in Gaza (1936)':     fetch_series('Eyeless in Gaza',     start_year=1900),
}

huxley_events = {
    1921: 'Crome Yellow published',
    1928: 'Point Counter Point published',
    1932: 'Brave New World\npublished',
    1936: 'Eyeless in Gaza published'
}

export_for_web(huxley_works, huxley_events, './articles/data/brave-new-world/huxley-works.json')

In [40]:
dystopian_classics = {
    'Brave New World (Huxley, 1932)':          fetch_series('Brave New World',    start_year=1900),
    'Lord of the Flies (Golding, 1954)':     fetch_series('Lord of the Flies', start_year=1900),
    'Fahrenheit 451 (Bradbury, 1953)':         fetch_series('Fahrenheit 451',     start_year=1900),
}

dystopian_events = {
    1932: 'Brave New World\npublished',
    1953: 'Fahrenheit 451 published',
    1954: 'Lord of the Flied published'
}

export_for_web(dystopian_classics, dystopian_events, './articles/data/brave-new-world/dystopian-classics.json')

Crime and Punishment

In [4]:
cp_titles = {
    'English': 'Crime and Punishment',
    'French':  'Crime et Chatiment',
    'German': 'Verbrechen und Strafe',
    'German': 'Schuld und Sühne',
    'Italian': 'Delitto e castigo',
    'Russian': 'Преступление и Наказание',
    'Spanish': 'Crimen y castigo',
}
cp_events = {
    1866: 'C&P published (serialized)',
    1881: 'Dostoevsky dies',
    1886: 'First English translation',
    1914: 'WWI begins',
    1917: 'Russian Revolution',
    1935: 'First Hollywood',
    1939: 'WWII begins',
    1953: 'Stalin dies',
    1956: 'Khrushchev speech',
    1969: 'Soviet film adaptation',
    1989: 'Berlin Wall falls',
    1991: 'Soviet Union dissolves',
    2002: 'BBC TV adaptation',
}
cp_lang_data = fetch_across_languages(cp_titles)
export_for_web(cp_lang_data, cp_events, './articles/data/crime-and-punishment/candp.json' )

In [5]:
dostoevsky_works = {
    'Crime and Punishment':   fetch_series('Crime and Punishment'),
    'The Brothers Karamazov': fetch_series('The Brothers Karamazov'),
    'The Idiot':              fetch_series('The Idiot'),
    'Notes from Underground': fetch_series('Notes from Underground'),
    'The Possessed':          fetch_series('The Possessed'),
}

export_for_web(dostoevsky_works, cp_events, './articles/data/crime-and-punishment/other_novels.json' )

In [6]:
crime_comparison = {
    'Crime and Punishment (Dostoevsky, 1866)':         fetch_series('Crime and Punishment'),
    'In Cold Blood (Capote, 1966)':                    fetch_series('In Cold Blood'),
    'The Da Vinci Code (Brown, 2003)':                 fetch_series('The Da Vinci Code'),
    'The Girl with the Dragon Tattoo (Larsson, 2005)': fetch_series('The Girl with the Dragon Tattoo'),
    'Gone Girl (Flynn, 2012)':                         fetch_series('Gone Girl'),
}

export_for_web(crime_comparison, cp_events, './articles/data/crime-and-punishment/same_genre.json' )